In [1]:
# ARC-AGI-3 feature engineering — kernel offline (save-and-run, sin internet)
import os, subprocess, sys
from pathlib import Path

COMP_ROOT = None
for dirpath, dirnames, _ in os.walk("/kaggle/input"):
    if "environment_files" in dirnames:
        COMP_ROOT = Path(dirpath)
        break
assert COMP_ROOT is not None, "no se encontró environment_files bajo /kaggle/input"
print("COMP_ROOT =", COMP_ROOT)

wheels = COMP_ROOT / "arc_agi_3_wheels"
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                f"--find-links={wheels}", "arcengine", "arc-agi"], check=True)
import arcengine, arc_agi
print("arcengine OK")


AssertionError: no se encontró environment_files bajo /kaggle/input

In [ ]:
# Reconstruye src/arc3 embebido (generado por build_features_notebook.py)
import os
os.makedirs('/kaggle/working/src/arc3', exist_ok=True)
SOURCES = {
 "src/arc3/__init__.py": "\"\"\"arc3: utilidades para ARC-AGI-3 (Kaggle arc-prize-2026-arc-agi-3).\n\nM\u00f3dulos:\n  env      -> descubrimiento y ejecuci\u00f3n local de environments (arcengine/arc_agi)\n  features -> feature engineering sobre frames 64x64 y transiciones (s, a, s')\n  probe    -> pol\u00edtica de sondeo que genera el dataset de features por juego\n\"\"\"\n\nfrom .features import (\n    connected_components,\n    grid_features,\n    frame_to_grid,\n    transition_features,\n)\n\n__all__ = [\n    \"connected_components\",\n    \"grid_features\",\n    \"frame_to_grid\",\n    \"transition_features\",\n]\n",
 "src/arc3/env.py": "\"\"\"Descubrimiento y ejecuci\u00f3n local de environments ARC-AGI-3.\n\nEnvuelve arc_agi.LocalEnvironmentWrapper para jugar los juegos de\nenvironment_files/ sin API remota (igual que har\u00e1 el rerun de Kaggle offline).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nfrom pathlib import Path\nfrom typing import Any, Optional\n\nfrom arc_agi.local_wrapper import LocalEnvironmentWrapper\nfrom arc_agi.models import EnvironmentInfo\nfrom arcengine import FrameDataRaw, GameAction\n\nlogger = logging.getLogger(\"arc3\")\n\n\ndef discover_environments(env_root: Path) -> list[EnvironmentInfo]:\n    \"\"\"Lista los environments locales a partir de environment_files/<game>/<hash>/metadata.json.\"\"\"\n    infos: list[EnvironmentInfo] = []\n    for meta_path in sorted(env_root.glob(\"*/*/metadata.json\")):\n        meta = json.loads(meta_path.read_text(encoding=\"utf-8\"))\n        # local_dir del metadata es relativo al root del dataset; usamos el real.\n        meta[\"local_dir\"] = str(meta_path.parent)\n        infos.append(EnvironmentInfo.model_validate(meta))\n    return infos\n\n\nclass LocalGame:\n    \"\"\"Sesi\u00f3n de un juego local: reset/step con FrameDataRaw.\"\"\"\n\n    def __init__(self, info: EnvironmentInfo, seed: int = 0) -> None:\n        self.info = info\n        self.env = LocalEnvironmentWrapper(\n            environment_info=info,\n            logger=logger,\n            scorecard_id=\"local-probe\",\n            seed=seed,\n            save_recording=False,\n        )\n\n    def reset(self) -> Optional[FrameDataRaw]:\n        return self.env.reset()\n\n    def step(\n        self, action: GameAction, x: Optional[int] = None, y: Optional[int] = None\n    ) -> Optional[FrameDataRaw]:\n        data: dict[str, Any] = {\"game_id\": self.info.game_id}\n        if action.is_complex():\n            data[\"x\"] = int(x or 0)\n            data[\"y\"] = int(y or 0)\n        return self.env.step(action, data=data)\n",
 "src/arc3/features.py": "\"\"\"Feature engineering para frames de ARC-AGI-3.\n\nLos frames son grids 64x64 con colores 0..15. Aqu\u00ed se computan:\n  - features por frame (histograma de color, objetos, simetr\u00edas, bordes, entrop\u00eda)\n  - features de transici\u00f3n (s, a, s'): p\u00edxeles cambiados, bbox del cambio,\n    deltas por color y detecci\u00f3n de traslaci\u00f3n (vector de movimiento)\n\nTodo en numpy puro (sin scipy) para poder correr offline en Kaggle sin deps extra.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom collections import deque\nfrom typing import Any, Optional, Sequence\n\nimport numpy as np\n\nN_COLORS = 16\nGRID = 64\n# Desplazamientos m\u00e1ximos a testear al detectar traslaci\u00f3n de objetos entre frames.\nMAX_SHIFT = 8\n\n\ndef frame_to_grid(frame: Any) -> np.ndarray:\n    \"\"\"Convierte FrameData.frame (lista de grids; puede traer varios por animaci\u00f3n)\n    al \u00faltimo grid como np.ndarray (64, 64) int8.\"\"\"\n    if frame is None or len(frame) == 0:\n        return np.zeros((GRID, GRID), dtype=np.int8)\n    last = frame[-1]\n    return np.asarray(last, dtype=np.int8)\n\n\ndef connected_components(\n    grid: np.ndarray, background: Optional[int] = None\n) -> list[dict[str, Any]]:\n    \"\"\"Componentes conexas 4-conectadas de celdas del mismo color (ignora el fondo).\n\n    Devuelve una lista de objetos: color, size, bbox (y0, x0, y1, x1), centroid.\n    BFS puro en python: el grid es 64x64, es barato.\n    \"\"\"\n    h, w = grid.shape\n    if background is None:\n        background = int(np.bincount(grid.ravel(), minlength=N_COLORS).argmax())\n    seen = np.zeros((h, w), dtype=bool)\n    objects: list[dict[str, Any]] = []\n    for y in range(h):\n        for x in range(w):\n            if seen[y, x] or grid[y, x] == background:\n                continue\n            color = int(grid[y, x])\n            q = deque([(y, x)])\n            seen[y, x] = True\n            cells = []\n            while q:\n                cy, cx = q.popleft()\n                cells.append((cy, cx))\n                for ny, nx in ((cy - 1, cx), (cy + 1, cx), (cy, cx - 1), (cy, cx + 1)):\n                    if 0 <= ny < h and 0 <= nx < w and not seen[ny, nx] and grid[ny, nx] == color:\n                        seen[ny, nx] = True\n                        q.append((ny, nx))\n            ys = [c[0] for c in cells]\n            xs = [c[1] for c in cells]\n            objects.append(\n                {\n                    \"color\": color,\n                    \"size\": len(cells),\n                    \"bbox\": (min(ys), min(xs), max(ys), max(xs)),\n                    \"centroid\": (float(np.mean(ys)), float(np.mean(xs))),\n                }\n            )\n    objects.sort(key=lambda o: -o[\"size\"])\n    return objects\n\n\ndef _edge_density(grid: np.ndarray) -> float:\n    \"\"\"Fracci\u00f3n de pares vecinos (4-conn) con colores distintos: mide 'estructura'.\"\"\"\n    dh = grid[:, 1:] != grid[:, :-1]\n    dv = grid[1:, :] != grid[:-1, :]\n    return float((dh.sum() + dv.sum()) / (dh.size + dv.size))\n\n\ndef _entropy(counts: np.ndarray) -> float:\n    p = counts[counts > 0].astype(np.float64)\n    p /= p.sum()\n    return float(-(p * np.log2(p)).sum())\n\n\ndef grid_features(grid: np.ndarray, max_objects: int = 8) -> dict[str, Any]:\n    \"\"\"Features escalares de un grid 64x64.\"\"\"\n    counts = np.bincount(grid.ravel(), minlength=N_COLORS)[:N_COLORS]\n    background = int(counts.argmax())\n    objects = connected_components(grid, background)\n    feats: dict[str, Any] = {\n        \"background\": background,\n        \"n_colors\": int((counts > 0).sum()),\n        \"color_entropy\": _entropy(counts),\n        \"edge_density\": _edge_density(grid),\n        \"sym_h\": float((grid == grid[:, ::-1]).mean()),  # simetr\u00eda izquierda-derecha\n        \"sym_v\": float((grid == grid[::-1, :]).mean()),  # simetr\u00eda arriba-abajo\n        \"n_objects\": len(objects),\n    }\n    for c in range(N_COLORS):\n        feats[f\"color_{c}\"] = int(counts[c])\n    for i in range(max_objects):\n        if i < len(objects):\n            o = objects[i]\n            y0, x0, y1, x1 = o[\"bbox\"]\n            feats[f\"obj{i}_color\"] = o[\"color\"]\n            feats[f\"obj{i}_size\"] = o[\"size\"]\n            feats[f\"obj{i}_cy\"], feats[f\"obj{i}_cx\"] = o[\"centroid\"]\n            feats[f\"obj{i}_h\"], feats[f\"obj{i}_w\"] = y1 - y0 + 1, x1 - x0 + 1\n        else:\n            feats[f\"obj{i}_color\"] = -1\n            feats[f\"obj{i}_size\"] = 0\n            feats[f\"obj{i}_cy\"] = feats[f\"obj{i}_cx\"] = -1.0\n            feats[f\"obj{i}_h\"] = feats[f\"obj{i}_w\"] = 0\n    return feats\n\n\ndef _detect_translation(prev: np.ndarray, nxt: np.ndarray, diff: np.ndarray) -> tuple[int, int, float]:\n    \"\"\"Busca el shift (dy, dx) que mejor explica el cambio como traslaci\u00f3n.\n\n    Solo mira la regi\u00f3n cambiada: si nxt == shift(prev) sobre esa regi\u00f3n, hay\n    movimiento de un objeto. Devuelve (dy, dx, score) con score en [0, 1].\n    \"\"\"\n    ys, xs = np.nonzero(diff)\n    if len(ys) == 0:\n        return 0, 0, 0.0\n    best = (0, 0, 0.0)\n    for dy in range(-MAX_SHIFT, MAX_SHIFT + 1):\n        for dx in range(-MAX_SHIFT, MAX_SHIFT + 1):\n            if dy == 0 and dx == 0:\n                continue\n            sy, sx = ys - dy, xs - dx\n            ok = (sy >= 0) & (sy < GRID) & (sx >= 0) & (sx < GRID)\n            if not ok.any():\n                continue\n            match = float((nxt[ys[ok], xs[ok]] == prev[sy[ok], sx[ok]]).mean())\n            if match > best[2]:\n                best = (dy, dx, match)\n    return best\n\n\ndef transition_features(prev_grid: np.ndarray, next_grid: np.ndarray) -> dict[str, Any]:\n    \"\"\"Features del cambio entre dos frames consecutivos.\"\"\"\n    diff = prev_grid != next_grid\n    n_changed = int(diff.sum())\n    feats: dict[str, Any] = {\"n_changed\": n_changed}\n    if n_changed == 0:\n        feats.update(\n            {\"chg_y0\": -1, \"chg_x0\": -1, \"chg_h\": 0, \"chg_w\": 0,\n             \"chg_area_frac\": 0.0, \"move_dy\": 0, \"move_dx\": 0, \"move_score\": 0.0,\n             \"colors_gained\": 0, \"colors_lost\": 0}\n        )\n        return feats\n    ys, xs = np.nonzero(diff)\n    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()\n    feats[\"chg_y0\"], feats[\"chg_x0\"] = int(y0), int(x0)\n    feats[\"chg_h\"], feats[\"chg_w\"] = int(y1 - y0 + 1), int(x1 - x0 + 1)\n    feats[\"chg_area_frac\"] = float(n_changed / diff.size)\n    prev_counts = np.bincount(prev_grid.ravel(), minlength=N_COLORS)[:N_COLORS]\n    next_counts = np.bincount(next_grid.ravel(), minlength=N_COLORS)[:N_COLORS]\n    delta = next_counts.astype(int) - prev_counts.astype(int)\n    feats[\"colors_gained\"] = int((delta > 0).sum())\n    feats[\"colors_lost\"] = int((delta < 0).sum())\n    dy, dx, score = _detect_translation(prev_grid, next_grid, diff)\n    feats[\"move_dy\"], feats[\"move_dx\"], feats[\"move_score\"] = dy, dx, score\n    return feats\n\n\ndef action_effect_summary(rows: Sequence[dict[str, Any]]) -> list[dict[str, Any]]:\n    \"\"\"Resumen por acci\u00f3n a partir de filas de transici\u00f3n: \u00bfqu\u00e9 acciones 'hacen algo'?\n\n    Cada fila debe traer: action_id, n_changed, level_up (bool), game_over (bool).\n    \"\"\"\n    out: list[dict[str, Any]] = []\n    by_action: dict[int, list[dict[str, Any]]] = {}\n    for r in rows:\n        by_action.setdefault(int(r[\"action_id\"]), []).append(r)\n    for action_id, rs in sorted(by_action.items()):\n        n = len(rs)\n        out.append(\n            {\n                \"action_id\": action_id,\n                \"n_uses\": n,\n                \"p_change\": float(np.mean([r[\"n_changed\"] > 0 for r in rs])),\n                \"avg_pixels_changed\": float(np.mean([r[\"n_changed\"] for r in rs])),\n                \"p_level_up\": float(np.mean([bool(r.get(\"level_up\")) for r in rs])),\n                \"p_game_over\": float(np.mean([bool(r.get(\"game_over\")) for r in rs])),\n            }\n        )\n    return out\n",
 "src/arc3/probe.py": "\"\"\"Pol\u00edtica de sondeo: juega cada environment y produce el dataset de features.\n\nEstrategia por juego:\n  1. RESET y features del frame inicial.\n  2. Round-robin sobre las acciones simples disponibles (ACTION1..5, 7) para\n     perfilar qu\u00e9 hace cada una (\u00bfcambia el frame?, \u00bfmueve un objeto?, \u00bfsube nivel?).\n  3. Sondeo de ACTION6 (click x,y) sobre una malla gruesa de puntos, para mapear\n     regiones interactivas.\n  4. Si el juego llega a GAME_OVER se hace RESET y se contin\u00faa hasta agotar budget.\n\nCada paso emite una fila con features de transici\u00f3n + features del frame resultante.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport random\nimport time\nfrom typing import Any, Optional\n\nfrom arcengine import FrameDataRaw, GameAction, GameState\n\nfrom .env import LocalGame\nfrom .features import frame_to_grid, grid_features, transition_features\n\nSIMPLE_ACTIONS = [\n    GameAction.ACTION1,\n    GameAction.ACTION2,\n    GameAction.ACTION3,\n    GameAction.ACTION4,\n    GameAction.ACTION5,\n    GameAction.ACTION7,\n]\n\n\ndef _click_grid(n: int = 8) -> list[tuple[int, int]]:\n    \"\"\"Malla n x n de puntos (x, y) centrados en tiles de 64/n.\"\"\"\n    step = 64 // n\n    half = step // 2\n    return [(x * step + half, y * step + half) for y in range(n) for x in range(n)]\n\n\ndef probe_game(\n    game: LocalGame,\n    budget: int = 300,\n    click_grid_n: int = 8,\n    seed: int = 0,\n    time_limit_s: Optional[float] = None,\n) -> list[dict[str, Any]]:\n    \"\"\"Sondea un juego y devuelve filas de features (una por acci\u00f3n ejecutada).\"\"\"\n    rng = random.Random(seed)\n    rows: list[dict[str, Any]] = []\n    t0 = time.time()\n\n    frame = game.reset()\n    if frame is None:\n        return rows\n    prev_grid = frame_to_grid(frame.frame)\n    prev_levels = frame.levels_completed\n\n    clicks = _click_grid(click_grid_n)\n    rng.shuffle(clicks)\n    click_i = 0\n    step_i = 0\n\n    while step_i < budget:\n        if time_limit_s is not None and time.time() - t0 > time_limit_s:\n            break\n        avail = frame.available_actions or []\n        simple = [a for a in SIMPLE_ACTIONS if not avail or a.value in avail]\n        use_click = (GameAction.ACTION6.value in avail or not avail) and (\n            not simple or step_i % 3 == 2\n        )\n\n        x = y = None\n        if use_click and click_i < len(clicks):\n            action = GameAction.ACTION6\n            x, y = clicks[click_i]\n            click_i += 1\n        elif simple:\n            action = simple[step_i % len(simple)]\n        elif GameAction.ACTION6.value in avail:\n            action = GameAction.ACTION6\n            x, y = rng.randrange(64), rng.randrange(64)\n        else:\n            break\n\n        nxt = game.step(action, x=x, y=y)\n        step_i += 1\n        if nxt is None:\n            continue\n\n        next_grid = frame_to_grid(nxt.frame)\n        row: dict[str, Any] = {\n            \"game_id\": game.info.game_id,\n            \"step\": step_i,\n            \"action_id\": action.value,\n            \"click_x\": -1 if x is None else x,\n            \"click_y\": -1 if y is None else y,\n            \"state\": nxt.state.value,\n            \"levels_completed\": nxt.levels_completed,\n            \"win_levels\": nxt.win_levels,\n            \"level_up\": nxt.levels_completed > prev_levels,\n            \"game_over\": nxt.state == GameState.GAME_OVER,\n            \"win\": nxt.state == GameState.WIN,\n        }\n        row.update(transition_features(prev_grid, next_grid))\n        row.update({f\"nf_{k}\": v for k, v in grid_features(next_grid).items()})\n        rows.append(row)\n\n        prev_levels = nxt.levels_completed\n        prev_grid = next_grid\n        frame = nxt\n\n        if nxt.state in (GameState.GAME_OVER, GameState.WIN):\n            frame = game.reset() or frame\n            prev_grid = frame_to_grid(frame.frame)\n            prev_levels = frame.levels_completed\n\n    return rows\n"
}
for path, code in SOURCES.items():
    with open('/kaggle/working/' + path, 'w', encoding='utf-8') as f:
        f.write(code)
print('src/arc3 reconstruido:', list(SOURCES))


In [ ]:
import sys, time
sys.path.insert(0, "/kaggle/working/src")
import pandas as pd
from pathlib import Path
from arc3.env import LocalGame, discover_environments
from arc3.features import action_effect_summary
from arc3.probe import probe_game

BUDGET = int(os.environ.get("ARC3_BUDGET", "500"))
TIME_LIMIT_S = float(os.environ.get("ARC3_TIME_LIMIT", "240"))
SEED = 0

out_dir = Path("/kaggle/working/features_out"); out_dir.mkdir(parents=True, exist_ok=True)
infos = discover_environments(COMP_ROOT / "environment_files")
print(len(infos), "environments")

all_rows, action_rows, game_rows = [], [], []
for info in infos:
    t0 = time.time()
    try:
        rows = probe_game(LocalGame(info, seed=SEED), budget=BUDGET, seed=SEED,
                          time_limit_s=TIME_LIMIT_S)
    except Exception as e:
        print("ERROR", info.game_id, e); continue
    all_rows.extend(rows)
    for s in action_effect_summary(rows):
        action_rows.append({"game_id": info.game_id, **s})
    game_rows.append({
        "game_id": info.game_id, "tags": ",".join(info.tags or []),
        "n_steps": len(rows),
        "max_levels_completed": max((r["levels_completed"] for r in rows), default=0),
        "win_levels": rows[-1]["win_levels"] if rows else 0,
        "n_game_overs": sum(r["game_over"] for r in rows),
        "seconds": round(time.time() - t0, 2)})
    print(game_rows[-1])

pd.DataFrame(all_rows).to_parquet(out_dir / "transitions.parquet", index=False)
pd.DataFrame(action_rows).to_csv(out_dir / "action_summary.csv", index=False)
pd.DataFrame(game_rows).to_csv(out_dir / "games_summary.csv", index=False)
print("filas:", len(all_rows))


In [ ]:
summary = pd.read_csv("/kaggle/working/features_out/games_summary.csv")
actions = pd.read_csv("/kaggle/working/features_out/action_summary.csv")
print("=== juegos ==="); print(summary.to_string(index=False))
print()
print("=== acciones con efecto (p_change > 0.2) ===")
print(actions[actions.p_change > 0.2].to_string(index=False))
